# 08 — Triangulation: where the three disagree, and what that costs

Three estimates of what each channel returns, and they do not agree.

**Attribution** is available daily, for every channel, at no cost, and is biased by
however much organic demand flows through that channel's last click.
**Experiments** are unbiased and are the only thing here that is, but cost weeks of
switched-off spend for one channel at a time. **A media-mix model** covers every
channel at once and is the only one that can say what the *next* dirham buys,
resting on assumptions about functional form that its own data cannot verify.

Reconciliation here does not mean averaging them. Averaging a biased estimator
with an unbiased one produces a biased estimator with a smaller variance, which is
worse to hand a decision-maker than either input because it looks more trustworthy
than it is.

In [ ]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [ ]:
triangulation = read_metric("triangulation", METRICS)
allocator = read_metric("allocator", METRICS)
print(triangulation["method_notes"]["why_not_average"])
print()
print(triangulation["method_notes"]["linear_versus_curved"])

## The three estimates, per channel

The divergence score is the coefficient of variation across the available
estimates. A high score is the signal a marketer should act on: it says the methods
are telling different stories about this channel, and that no amount of dashboard
polish will settle which is right without an experiment.

In [ ]:
rows = []
for channel, entry in triangulation["comparison"]["channels"].items():
    rows.append(
        {
            "channel": channel,
            "true": round(entry["true_roi"], 3),
            "attribution": round(entry["attribution"]["estimate"], 3),
            "mmm": round(entry["mmm"]["estimate"], 3) if entry["mmm"]["estimate"] else None,
            "experiment": round(entry["experiment"]["estimate"], 3)
            if entry["experiment"].get("estimate")
            else None,
            "divergence": round(entry["divergence"], 3),
            "closest": entry["closest_method"],
        }
    )
show(pd.DataFrame(rows))
print("most divergent :", triangulation["comparison"]["summary"]["most_divergent_channel"])
print("least divergent:", triangulation["comparison"]["summary"]["least_divergent_channel"])

## Turning each into a budget

Every allocation runs under identical governance constraints — a floor and a cap
per channel. Without them a scalar ROI table sends the whole budget to one channel,
and the comparison would be against a caricature no marketing organisation
resembles.

Only the media-mix model produces a *curve*, so only it can allocate on curvature.
An experiment and an attribution report each return a single number per channel and
say nothing about the next dirham, so a planner holding only those must treat
returns as constant.

In [ ]:
print(json.dumps(allocator["governance"], indent=2))
print()
rows = []
for name, entry in allocator["allocations"].items():
    row = {"allocation": name}
    row.update({channel: round(share, 3) for channel, share in entry["shares"].items()})
    row["revenue under truth"] = round(entry["revenue_under_truth"], 0)
    row["shortfall"] = round(entry["shortfall_share"], 4)
    rows.append(row)
show(pd.DataFrame(rows).sort_values("shortfall"))

## The answer

Every allocation is scored against the *same* true response curves, so what is
compared is the consequence of believing an estimator rather than the estimator's
own opinion of itself. The benchmark is the allocation built from the true curves —
a ceiling nobody can reach, and the right benchmark precisely because it separates
"this estimator is wrong" from "this problem is hard".

In [ ]:
headline = triangulation["headline"]
for key, value in headline.items():
    if key != "statement":
        print(f"{key:44s} {value}")
print()
print(headline["statement"])

## The same figure at the scale the brief describes

A stated scenario, not a measurement. Olist trades in 2017-18 Brazilian reais and
this project makes no claim about any exchange rate; the shortfall is a share of
budget, so it scales.

In [ ]:
scenario = triangulation["scenario_in_aed"]
print(scenario["caveat"])
print()
print(scenario["on_a_one_million_aed_budget"]["note"])

## And the lifetime-value consequence

If lifetime value is proportional to first-order value, the CLV-weighted
reallocation is the same reallocation.

In [ ]:
if triangulation.get("clv_consequence"):
    for key, value in triangulation["clv_consequence"].items():
        print(f"{key}:\n  {value}\n")
else:
    print("metrics/clv.json was not present when this was generated.")

---

**What would falsify this.** The ordering of attribution bias across channels is
pre-registered rather than measured, so a business whose brand search genuinely
drives incremental demand would invert the headline. The media-mix result is
conditional on a signal share of roughly 17% of detrended variance; a quieter media
plan is a harder recovery problem and the reported errors would grow. And the
allocation comparison assumes the governance floors and caps, without which every
scalar-ROI allocation collapses to a corner.